# ETL FactVentas

Este notebook carga una fila por detalle de venta. La carga es incremental y conserva los registros ya existentes.

## 1. Importar librerías

Usaremos pandas para transformar los datos, SQLAlchemy para conectarnos a SQL Server y python-dotenv para leer la configuración.

In [1]:
import os
from pathlib import Path
from urllib.parse import quote_plus

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

## 2. Configurar conexiones

Las credenciales se leen desde `etl/.env`; no se escriben ni se muestran en el notebook. Se comprueba que estén disponibles antes de crear las conexiones.

In [3]:
# Funciona tanto si Jupyter inicia en la raíz del proyecto como en etl/.
ruta_env = Path("../.env").resolve()
load_dotenv(dotenv_path=ruta_env)

variables_requeridas = [
    "SQLSERVER_DRIVER", "SQLSERVER_SERVIDOR", "SQLSERVER_BASE_DATOS",
    "SQLSERVER_USUARIO", "SQLSERVER_PASSWORD", "SQLSERVER_ESQUEMA",
    "SQLSERVER_DRIVER_DW", "SQLSERVER_SERVIDOR_DW",
    "SQLSERVER_BASE_DATOS_DW", "SQLSERVER_USUARIO_DW",
    "SQLSERVER_PASSWORD_DW", "SQLSERVER_ESQUEMA_DW"
]
variables_faltantes = [
    nombre for nombre in variables_requeridas if not os.getenv(nombre)
]
if variables_faltantes:
    raise ValueError(
        "Faltan variables de conexión en etl/.env: "
        + ", ".join(variables_faltantes)
    )

cadena_stage = quote_plus(
    f"DRIVER={{{os.getenv('SQLSERVER_DRIVER')}}};"
    f"SERVER={os.getenv('SQLSERVER_SERVIDOR')};"
    f"DATABASE={os.getenv('SQLSERVER_BASE_DATOS')};"
    f"UID={os.getenv('SQLSERVER_USUARIO')};"
    f"PWD={os.getenv('SQLSERVER_PASSWORD')};"
    "TrustServerCertificate=yes;"
)
engine_stage = create_engine(
    f"mssql+pyodbc:///?odbc_connect={cadena_stage}"
)

cadena_dw = quote_plus(
    f"DRIVER={{{os.getenv('SQLSERVER_DRIVER_DW')}}};"
    f"SERVER={os.getenv('SQLSERVER_SERVIDOR_DW')};"
    f"DATABASE={os.getenv('SQLSERVER_BASE_DATOS_DW')};"
    f"UID={os.getenv('SQLSERVER_USUARIO_DW')};"
    f"PWD={os.getenv('SQLSERVER_PASSWORD_DW')};"
    "TrustServerCertificate=yes;"
)
engine_dw = create_engine(
    f"mssql+pyodbc:///?odbc_connect={cadena_dw}"
)

esquema_stage = os.getenv("SQLSERVER_ESQUEMA")
esquema_dw = os.getenv("SQLSERVER_ESQUEMA_DW")
print("Conexiones configuradas para Stage y Data Warehouse.")

Conexiones configuradas para Stage y Data Warehouse.


## 3. Extraer ventas y detalles

Unimos `venta` y `detalle_venta` por `venta_id`. La unión externa permite detectar y reportar detalles que no tengan una venta asociada, en vez de descartarlos silenciosamente.

In [3]:
consulta_ventas = f"""
SELECT
    d.detalle_venta_id,
    d.venta_id,
    d.producto_id,
    d.cantidad,
    d.precio_unitario,
    d.costo_unitario,
    d.descuento,
    v.cliente_id,
    v.fecha_venta
FROM {esquema_stage}.detalle_venta AS d
LEFT JOIN {esquema_stage}.venta AS v
    ON d.venta_id = v.venta_id
"""

df_ventas = pd.read_sql(consulta_ventas, engine_stage)
registros_extraidos = len(df_ventas)
print("Registros extraídos:", registros_extraidos)
df_ventas.head()

Registros extraídos: 21962


,detalle_venta_id,venta_id,producto_id,cantidad,precio_unitario,costo_unitario,descuento,cliente_id,fecha_venta
0,1,1,55,1,442.90,309.60,22.15,663.0,2023-10-08 17:44:13
1,2,2,24,1,375.95,211.70,56.39,NaN,2023-07-16 13:46:24
2,3,2,20,1,309.75,197.65,0.00,NaN,2023-07-16 13:46:24
3,4,3,58,3,250.00,140.00,0.00,197.0,2023-07-20 22:42:22
4,5,3,39,1,86.40,60.30,8.64,197.0,2023-07-20 22:42:22


## 4. Validar y transformar detalles

Cada detalle debe tener una clave y los datos necesarios para calcular sus importes y relacionarlo con las dimensiones. Los detalles incompletos se contabilizan y se excluyen porque no pueden cargarse en el hecho; el descuento nulo se interpreta como cero. Las claves duplicadas siguen deteniendo el proceso.

In [4]:
columnas_obligatorias = [
    "detalle_venta_id", "venta_id", "producto_id", "cantidad",
    "precio_unitario", "costo_unitario", "cliente_id", "fecha_venta"
]
filas_incompletas = df_ventas[columnas_obligatorias].isna().any(axis=1)
registros_rechazados = int(filas_incompletas.sum())
if registros_rechazados:
    print(
        "Detalles excluidos por datos obligatorios faltantes:",
        registros_rechazados
    )
    print("Campos faltantes por columna:")
    print(df_ventas.loc[filas_incompletas, columnas_obligatorias].isna().sum())
    df_ventas = df_ventas.loc[~filas_incompletas].copy()

duplicados_detalle = df_ventas["detalle_venta_id"].duplicated(keep=False)
if duplicados_detalle.any():
    raise ValueError(
        "Stage contiene claves detalle_venta_id repetidas; "
        "no es posible asignar una fila inequívoca por detalle."
    )

df_ventas["fecha_venta"] = pd.to_datetime(
    df_ventas["fecha_venta"], errors="raise"
)
df_ventas["fecha_sk"] = df_ventas["fecha_venta"].dt.strftime("%Y%m%d").astype("int64")
df_ventas["descuento"] = pd.to_numeric(
    df_ventas["descuento"], errors="raise"
).fillna(0)

for columna in ["cantidad", "precio_unitario", "costo_unitario"]:
    df_ventas[columna] = pd.to_numeric(df_ventas[columna], errors="raise")

if (df_ventas["cantidad"] <= 0).any():
    raise ValueError("Stage contiene detalles con cantidad menor o igual a cero.")

print("Detalles válidos transformados:", len(df_ventas))

Detalles excluidos por datos obligatorios faltantes: 1564
Campos faltantes por columna:
detalle_venta_id       0
venta_id               0
producto_id            0
cantidad               0
precio_unitario        0
costo_unitario         0
cliente_id          1564
fecha_venta            0
dtype: int64


Detalles válidos transformados: 20398


## 5. Detectar detalles pendientes

Comparamos `detalle_venta_id` con la clave de origen guardada en FactVentas. Solo los detalles ausentes en el DW seguirán hacia la carga.

In [5]:
df_detalles_existentes = pd.read_sql(
    f"SELECT detalle_venta_id_origen FROM {esquema_dw}.FactVentas",
    engine_dw
)
registros_existentes = len(df_detalles_existentes)

df_ventas_nuevas = df_ventas[
    ~df_ventas["detalle_venta_id"].isin(
        df_detalles_existentes["detalle_venta_id_origen"]
    )
].copy()
registros_nuevos = len(df_ventas_nuevas)

print("Registros existentes en FactVentas:", registros_existentes)
print("Registros nuevos para cargar:", registros_nuevos)
print("Registros excluidos por datos faltantes:", registros_rechazados)

Registros existentes en FactVentas: 20398
Registros nuevos para cargar: 0
Registros excluidos por datos faltantes: 1564


## 6. Buscar claves sustitutas y calcular medidas

Las claves de cliente, producto y fecha se obtienen de sus dimensiones. El importe de venta es cantidad × precio unitario − descuento; el costo total es cantidad × costo unitario y la utilidad es la diferencia entre ambos.

In [6]:
columnas_fact = [
    "venta_id_origen", "detalle_venta_id_origen", "cliente_sk",
    "producto_sk", "fecha_sk", "cantidad", "precio_unitario",
    "costo_unitario", "descuento", "importe_venta", "costo_total",
    "utilidad"
]

if df_ventas_nuevas.empty:
    df_fact_ventas = pd.DataFrame(columns=columnas_fact)
else:
    df_clientes = pd.read_sql(
        f"SELECT cliente_id_origen, cliente_sk FROM {esquema_dw}.DimCliente",
        engine_dw
    )
    df_productos = pd.read_sql(
        f"SELECT producto_id_origen, producto_sk FROM {esquema_dw}.DimProducto",
        engine_dw
    )
    df_fechas = pd.read_sql(
        f"SELECT fecha_sk FROM {esquema_dw}.DimFecha",
        engine_dw
    )

    df_fact_ventas = df_ventas_nuevas.merge(
        df_clientes,
        left_on="cliente_id",
        right_on="cliente_id_origen",
        how="left",
        validate="many_to_one"
    ).merge(
        df_productos,
        left_on="producto_id",
        right_on="producto_id_origen",
        how="left",
        validate="many_to_one"
    ).merge(
        df_fechas,
        on="fecha_sk",
        how="left",
        validate="many_to_one",
        indicator="fecha_encontrada"
    )

    sin_cliente = df_fact_ventas["cliente_sk"].isna()
    sin_producto = df_fact_ventas["producto_sk"].isna()
    sin_fecha = df_fact_ventas["fecha_encontrada"] != "both"
    if sin_cliente.any() or sin_producto.any() or sin_fecha.any():
        raise ValueError(
            "Hay detalles nuevos sin correspondencia en las dimensiones: "
            f"clientes={int(sin_cliente.sum())}, "
            f"productos={int(sin_producto.sum())}, "
            f"fechas={int(sin_fecha.sum())}. Carga primero las dimensiones."
        )

    df_fact_ventas["importe_venta"] = (
        df_fact_ventas["cantidad"] * df_fact_ventas["precio_unitario"]
        - df_fact_ventas["descuento"]
    ).round(2)
    df_fact_ventas["costo_total"] = (
        df_fact_ventas["cantidad"] * df_fact_ventas["costo_unitario"]
    ).round(2)
    df_fact_ventas["utilidad"] = (
        df_fact_ventas["importe_venta"] - df_fact_ventas["costo_total"]
    ).round(2)

    df_fact_ventas = df_fact_ventas.rename(
        columns={
            "venta_id": "venta_id_origen",
            "detalle_venta_id": "detalle_venta_id_origen"
        }
    )[columnas_fact]

print("Filas preparadas para FactVentas:", len(df_fact_ventas))
df_fact_ventas.head()

Filas preparadas para FactVentas: 0


,venta_id_origen,detalle_venta_id_origen,cliente_sk,producto_sk,fecha_sk,cantidad,precio_unitario,costo_unitario,descuento,importe_venta,costo_total,utilidad


## 7. Cargar datos

Insertamos únicamente los detalles nuevos con `append`. SQL Server genera `venta_sk` porque es una columna `IDENTITY`; no la incluimos en los datos.

In [7]:
if not df_fact_ventas.empty:
    df_fact_ventas.to_sql(
        name="FactVentas",
        con=engine_dw,
        schema=esquema_dw,
        if_exists="append",
        index=False
    )
    registros_insertados = len(df_fact_ventas)
else:
    registros_insertados = 0

print("Registros insertados:", registros_insertados)

Registros insertados: 0


## 8. Validar resultados y cerrar conexiones

Revisamos los conteos del proceso. Una segunda ejecución sin cambios debe informar cero registros nuevos e insertados.

In [8]:
print("Registros extraídos:", registros_extraidos)
print("Registros existentes en FactVentas:", registros_existentes)
print("Registros nuevos:", registros_nuevos)
print("Registros insertados:", registros_insertados)
print("Registros excluidos por datos faltantes:", registros_rechazados)

engine_stage.dispose()
engine_dw.dispose()
print("Conexiones cerradas.")

Registros extraídos: 21962
Registros existentes en FactVentas: 20398
Registros nuevos: 0
Registros insertados: 0
Registros excluidos por datos faltantes: 1564
Conexiones cerradas.
